In [ ]:
"""
Teeworlds Bot Framework
========================
Reads game state (input/output variables) from a running Teeworlds process.

Two approaches are provided:
  1. MemoryReader  – reads raw process memory (Windows/Linux, requires pymem/ctypes)
  2. SnapshotClient – connects as a network client and parses game snapshots
                      (recommended: more stable, cross-platform, protocol-level)

Dependencies:
    pip install pymem     # for MemoryReader (Windows)
    pip install pywin32   # for SendInput on Windows
    No extra deps needed for SnapshotClient

Usage example at the bottom of this file.
"""

import struct
import socket
import time
import ctypes
import threading
from dataclasses import dataclass, field
from typing import Optional, List, Dict

# ─────────────────────────────────────────────────────────────────────────────
# DATA STRUCTURES  (mirrors Teeworlds source: src/game/gamecore.h)
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class Vec2:
    x: float = 0.0
    y: float = 0.0

    def __repr__(self):
        return f"Vec2({self.x:.1f}, {self.y:.1f})"


@dataclass
class PlayerInput:
    """
    Mirrors CNetObj_PlayerInput in Teeworlds source.
    These are the INPUT variables you send to control your tee.
    """
    direction: int = 0          # -1 = left, 0 = none, +1 = right
    target_x: int = 0           # mouse aim X (relative to player, world coords)
    target_y: int = 0           # mouse aim Y
    jump: int = 0               # 1 = jump pressed
    fire: int = 0               # fire tick counter (increment to fire)
    hook: int = 0               # 1 = hook pressed
    player_flags: int = 0       # PLAYERFLAG_CHATTING etc.
    wanted_weapon: int = 0      # 0=hammer,1=gun,2=shotgun,3=grenade,4=laser,5=ninja
    next_weapon: int = 0        # scroll up weapon
    prev_weapon: int = 0        # scroll down weapon


@dataclass
class PlayerState:
    """
    OUTPUT variables: the game state you read to make decisions.
    Mirrors CNetObj_Character in Teeworlds source.
    """
    # --- Identity ---
    client_id: int = -1
    name: str = ""
    team: int = 0               # 0=red, 1=blue, -1=spectator

    # --- Position & movement ---
    pos: Vec2 = field(default_factory=Vec2)
    vel: Vec2 = field(default_factory=Vec2)

    # --- Health / armor ---
    health: int = 0             # 0–10
    armor: int = 0              # 0–10
    alive: bool = False

    # --- Weapons ---
    active_weapon: int = 0      # current weapon index
    weapon_ammo: List[int] = field(default_factory=lambda: [0]*6)

    # --- Actions ---
    jumped: int = 0             # jump bits (double-jump state)
    hooked_player: int = -1     # client_id of hooked player, -1 if none
    hook_state: int = 0         # 0=idle,1=throwing,4=grabbed,5=retracted
    hook_pos: Vec2 = field(default_factory=Vec2)

    # --- Misc ---
    direction: int = 0
    angle: int = 0              # aim angle (fixed-point, /256.0 = radians)
    emote: int = 0
    attack_tick: int = 0        # last attack game tick


@dataclass
class GameState:
    """Top-level snapshot of the full game."""
    game_tick: int = 0
    game_flags: int = 0         # GAMEFLAG_TEAMS, GAMEFLAG_FLAGS, etc.

    # Scores
    score_limit: int = 0
    time_limit: int = 0
    warmup_timer: int = 0

    # CTF
    flag_carrier_red: int = -1
    flag_carrier_blue: int = -1

    # All players on the server
    players: Dict[int, PlayerState] = field(default_factory=dict)

    # Local player info
    local_client_id: int = -1

    @property
    def local_player(self) -> Optional[PlayerState]:
        return self.players.get(self.local_client_id)

    @property
    def enemies(self) -> List[PlayerState]:
        local = self.local_player
        if not local:
            return []
        return [p for cid, p in self.players.items()
                if cid != self.local_client_id and p.alive and p.team != local.team]

    @property
    def teammates(self) -> List[PlayerState]:
        local = self.local_player
        if not local:
            return []
        return [p for cid, p in self.players.items()
                if cid != self.local_client_id and p.alive and p.team == local.team]


# ─────────────────────────────────────────────────────────────────────────────
# APPROACH 1: MEMORY READER
# Reads game state directly from process memory.
# Offsets are for Teeworlds 0.7.x 64-bit. You may need to rescan with
# Cheat Engine if your version differs.
# ─────────────────────────────────────────────────────────────────────────────

class MemoryReader:
    """
    Reads Teeworlds internal state from process memory.
    
    ⚠ NOTE: Memory offsets change between builds. These are approximate
    starting points for Teeworlds 0.7.5 x64. Use Cheat Engine to find/verify
    offsets for your exact build by scanning for known values (e.g., health=10).
    
    Requires: pip install pymem
    """

    # Base offsets inside the teeworlds process (update per build)
    OFFSETS = {
        # These are EXAMPLE offsets – scan your own build with Cheat Engine
        # Pattern: CGameClient -> m_aClients[i] -> m_aCharacters[i]
        "game_tick":        0x00,   # int32 – current game tick
        "local_id":         0x04,   # int32 – our client ID
        "num_players":      0x08,   # int32

        # Per-player block size in m_aCharacters array
        "player_stride":    0xC0,   # bytes per CNetObj_Character

        # Offsets within each CNetObj_Character block
        "char_x":           0x00,   # int32 (fixed-point /32.0 = world units)
        "char_y":           0x04,
        "char_vel_x":       0x08,
        "char_vel_y":       0x0C,
        "char_angle":       0x10,   # int32
        "char_direction":   0x14,
        "char_jumped":      0x18,
        "char_hook_x":      0x1C,
        "char_hook_y":      0x20,
        "char_hook_state":  0x24,
        "char_hook_tee":    0x28,
        "char_health":      0x2C,
        "char_armor":       0x30,
        "char_ammo_count":  0x34,
        "char_weapon":      0x38,
        "char_emote":       0x3C,
        "char_attack_tick": 0x40,
    }

    def __init__(self, process_name: str = "teeworlds.exe"):
        self.process_name = process_name
        self.pm = None
        self.base_address = None
        self._connected = False

    def connect(self) -> bool:
        """Attach to the running Teeworlds process."""
        try:
            import pymem
            import pymem.process
            self.pm = pymem.Pymem(self.process_name)
            # Get base module address
            module = pymem.process.module_from_name(
                self.pm.process_handle, self.process_name
            )
            self.base_address = module.lpBaseOfDll
            self._connected = True
            print(f"[MemoryReader] Attached to {self.process_name} @ 0x{self.base_address:X}")
            return True
        except Exception as e:
            print(f"[MemoryReader] Failed to attach: {e}")
            return False

    def _read_int32(self, address: int) -> int:
        if not self._connected:
            return 0
        try:
            return self.pm.read_int(address)
        except Exception:
            return 0

    def _read_float(self, address: int) -> float:
        if not self._connected:
            return 0.0
        try:
            return self.pm.read_float(address)
        except Exception:
            return 0.0

    def read_player(self, client_id: int, char_base_addr: int) -> PlayerState:
        """Read a single player's CNetObj_Character from memory."""
        o = self.OFFSETS
        stride = o["player_stride"]
        base = char_base_addr + client_id * stride

        raw_x = self._read_int32(base + o["char_x"])
        raw_y = self._read_int32(base + o["char_y"])
        raw_vx = self._read_int32(base + o["char_vel_x"])
        raw_vy = self._read_int32(base + o["char_vel_y"])

        ps = PlayerState(
            client_id=client_id,
            pos=Vec2(raw_x / 32.0, raw_y / 32.0),   # fixed-point to world units
            vel=Vec2(raw_vx / 256.0, raw_vy / 256.0),
            angle=self._read_int32(base + o["char_angle"]),
            direction=self._read_int32(base + o["char_direction"]),
            jumped=self._read_int32(base + o["char_jumped"]),
            hook_pos=Vec2(
                self._read_int32(base + o["char_hook_x"]) / 32.0,
                self._read_int32(base + o["char_hook_y"]) / 32.0,
            ),
            hook_state=self._read_int32(base + o["char_hook_state"]),
            hooked_player=self._read_int32(base + o["char_hook_tee"]),
            health=self._read_int32(base + o["char_health"]),
            armor=self._read_int32(base + o["char_armor"]),
            active_weapon=self._read_int32(base + o["char_weapon"]),
            emote=self._read_int32(base + o["char_emote"]),
            attack_tick=self._read_int32(base + o["char_attack_tick"]),
            alive=self._read_int32(base + o["char_health"]) > 0,
        )
        return ps

    def read_game_state(self, snapshot_base_addr: int, char_base_addr: int) -> GameState:
        """Read the full game state from memory."""
        o = self.OFFSETS
        gs = GameState(
            game_tick=self._read_int32(snapshot_base_addr + o["game_tick"]),
            local_client_id=self._read_int32(snapshot_base_addr + o["local_id"]),
        )
        num_players = self._read_int32(snapshot_base_addr + o["num_players"])
        for i in range(min(num_players, 16)):
            ps = self.read_player(i, char_base_addr)
            gs.players[i] = ps
        return gs

    def send_input(self, inp: PlayerInput):
        """
        Inject input by writing to the input buffer in memory.
        Alternatively, use keyboard/mouse simulation (see InputSender below).
        """
        # TODO: write to CGameClient->m_Input at the correct offset
        # This is highly version-specific; use InputSender for safer injection.
        print(f"[MemoryWriter] Would send: {inp}")


# ─────────────────────────────────────────────────────────────────────────────
# APPROACH 2: NETWORK SNAPSHOT CLIENT (recommended)
# Connects to a server and parses snapshot packets that carry full game state.
# No memory hacking needed. Cross-platform.
# ─────────────────────────────────────────────────────────────────────────────

class TwInt:
    """Teeworlds variable-length integer codec."""

    @staticmethod
    def unpack(data: bytes, offset: int = 0):
        """Decode a Teeworlds variable-length int. Returns (value, new_offset)."""
        b = data[offset]; offset += 1
        sign = (b >> 6) & 1
        value = b & 0x3F
        shift = 6
        while b & 0x80:
            b = data[offset]; offset += 1
            value |= (b & 0x7F) << shift
            shift += 7
        if sign:
            value = -(value + 1)
        return value, offset

    @staticmethod
    def pack(value: int) -> bytes:
        """Encode an integer as Teeworlds variable-length int."""
        if value < 0:
            value = -(value + 1)
            sign = 1
        else:
            sign = 0
        result = []
        b = (sign << 6) | (value & 0x3F)
        value >>= 6
        while value:
            result.append(b | 0x80)
            b = value & 0x7F
            value >>= 7
        result.append(b)
        return bytes(result)


class SnapshotParser:
    """
    Parses Teeworlds snapshot data into GameState.
    Snapshots carry the full authoritative game state from the server.
    Item type IDs from src/game/gamecore.h
    """

    # NetObj type IDs
    NETOBJ_PLAYER_INPUT     = 1
    NETOBJ_PROJECTILE       = 2
    NETOBJ_LASER            = 3
    NETOBJ_PICKUP           = 4
    NETOBJ_FLAG             = 5
    NETOBJ_GAME_INFO        = 6
    NETOBJ_GAME_DATA        = 7
    NETOBJ_CHARACTER_CORE   = 8
    NETOBJ_CHARACTER        = 9
    NETOBJ_PLAYER_INFO      = 10
    NETOBJ_CLIENT_INFO      = 11
    NETOBJ_SPECTATOR_INFO   = 12

    def parse_character(self, data: bytes, offset: int, client_id: int) -> PlayerState:
        """Parse CNetObj_Character (22 int32 fields)."""
        ints = []
        for _ in range(22):
            v, offset = TwInt.unpack(data, offset)
            ints.append(v)

        # Field order from src/game/gamecore.h CNetObj_Character
        # tick, x, y, vel_x, vel_y, angle, direction, jumped,
        # hook_tick, hook_state, hook_tee, hook_x, hook_y,
        # health, armor, ammo_count, weapon, emote, attack_tick,
        # triggered_events (skip), player_flags, player_state
        ps = PlayerState(
            client_id=client_id,
            pos=Vec2(ints[1] / 32.0, ints[2] / 32.0),
            vel=Vec2(ints[3] / 256.0, ints[4] / 256.0),
            angle=ints[5],
            direction=ints[6],
            jumped=ints[7],
            hook_state=ints[9],
            hooked_player=ints[10],
            hook_pos=Vec2(ints[11] / 32.0, ints[12] / 32.0),
            health=ints[13],
            armor=ints[14],
            weapon_ammo=[ints[15]] + [0]*5,
            active_weapon=ints[16],
            emote=ints[17],
            attack_tick=ints[18],
            alive=ints[13] > 0,
        )
        return ps, offset

    def parse_game_data(self, data: bytes, offset: int) -> dict:
        """Parse CNetObj_GameData."""
        game_tick, offset   = TwInt.unpack(data, offset)
        game_flags, offset  = TwInt.unpack(data, offset)
        score_r, offset     = TwInt.unpack(data, offset)
        score_b, offset     = TwInt.unpack(data, offset)
        flag_r, offset      = TwInt.unpack(data, offset)
        flag_b, offset      = TwInt.unpack(data, offset)
        return {
            "game_tick": game_tick,
            "game_flags": game_flags,
            "score_red": score_r,
            "score_blue": score_b,
            "flag_carrier_red": flag_r,
            "flag_carrier_blue": flag_b,
        }, offset

    def parse_snapshot(self, raw: bytes, local_client_id: int = -1) -> GameState:
        """
        Parse a full snapshot payload into a GameState.
        raw: the raw snapshot bytes (after packet header is stripped).
        """
        gs = GameState(local_client_id=local_client_id)
        offset = 0
        try:
            num_items, offset = TwInt.unpack(raw, offset)
            for _ in range(num_items):
                type_and_id, offset = TwInt.unpack(raw, offset)
                item_type = (type_and_id >> 16) & 0xFFFF
                item_id   = type_and_id & 0xFFFF
                item_size, offset = TwInt.unpack(raw, offset)
                item_data = raw[offset:offset + item_size]
                offset   += item_size

                if item_type == self.NETOBJ_CHARACTER:
                    ps, _ = self.parse_character(item_data, 0, item_id)
                    gs.players[item_id] = ps

                elif item_type == self.NETOBJ_GAME_DATA:
                    gd, _ = self.parse_game_data(item_data, 0)
                    gs.game_tick = gd["game_tick"]
                    gs.game_flags = gd["game_flags"]
                    gs.flag_carrier_red = gd["flag_carrier_red"]
                    gs.flag_carrier_blue = gd["flag_carrier_blue"]
        except Exception as e:
            print(f"[SnapshotParser] Parse error at offset {offset}: {e}")
        return gs


# ─────────────────────────────────────────────────────────────────────────────
# INPUT SENDER
# Simulates keyboard/mouse to send input to the game window.
# Works without memory writing – just normal OS-level input simulation.
# ─────────────────────────────────────────────────────────────────────────────

try:
    import ctypes
    import ctypes.wintypes
    WINDOWS = True
except Exception:
    WINDOWS = False


class InputSender:
    """
    Sends keyboard and mouse input to the Teeworlds window.
    Uses OS-level input simulation (platform-aware).
    """

    # Key codes for Teeworlds default bindings
    KEY_LEFT    = 0x41  # A
    KEY_RIGHT   = 0x44  # D
    KEY_JUMP    = 0x20  # Space
    KEY_HOOK    = 0x01  # Left mouse (handled via mouse)
    KEY_FIRE    = 0x02  # Right mouse

    WEAPON_KEYS = {
        0: 0x31,  # 1 = Hammer
        1: 0x32,  # 2 = Gun
        2: 0x33,  # 3 = Shotgun
        3: 0x34,  # 4 = Grenade
        4: 0x35,  # 5 = Laser
        5: 0x36,  # 6 = Ninja
    }

    def __init__(self):
        self._held_keys = set()

    # ── Windows implementation ──────────────────────────────────────────────

    def _win_key_down(self, vk: int):
        if WINDOWS and vk not in self._held_keys:
            ctypes.windll.user32.keybd_event(vk, 0, 0, 0)
            self._held_keys.add(vk)

    def _win_key_up(self, vk: int):
        if WINDOWS and vk in self._held_keys:
            ctypes.windll.user32.keybd_event(vk, 0, 2, 0)  # KEYEVENTF_KEYUP
            self._held_keys.discard(vk)

    def _win_move_mouse(self, screen_x: int, screen_y: int):
        if WINDOWS:
            ctypes.windll.user32.SetCursorPos(screen_x, screen_y)

    def _win_mouse_click(self, button: str = "left"):
        if WINDOWS:
            if button == "left":
                ctypes.windll.user32.mouse_event(2, 0, 0, 0, 0)   # down
                time.sleep(0.02)
                ctypes.windll.user32.mouse_event(4, 0, 0, 0, 0)   # up
            elif button == "right":
                ctypes.windll.user32.mouse_event(8, 0, 0, 0, 0)
                time.sleep(0.02)
                ctypes.windll.user32.mouse_event(16, 0, 0, 0, 0)

    # ── Linux implementation (xdotool) ──────────────────────────────────────

    def _linux_key_down(self, key_name: str):
        import subprocess
        subprocess.Popen(["xdotool", "keydown", key_name])

    def _linux_key_up(self, key_name: str):
        import subprocess
        subprocess.Popen(["xdotool", "keyup", key_name])

    def _linux_move_mouse(self, x: int, y: int):
        import subprocess
        subprocess.Popen(["xdotool", "mousemove", str(x), str(y)])

    # ── High-level API ───────────────────────────────────────────────────────

    def apply_input(self, inp: PlayerInput, aim_screen_x: int, aim_screen_y: int):
        """Apply a PlayerInput to the game via OS input simulation."""
        import platform

        if platform.system() == "Windows":
            # Movement
            if inp.direction < 0:
                self._win_key_down(self.KEY_LEFT)
                self._win_key_up(self.KEY_RIGHT)
            elif inp.direction > 0:
                self._win_key_down(self.KEY_RIGHT)
                self._win_key_up(self.KEY_LEFT)
            else:
                self._win_key_up(self.KEY_LEFT)
                self._win_key_up(self.KEY_RIGHT)

            # Jump
            if inp.jump:
                self._win_key_down(self.KEY_JUMP)
            else:
                self._win_key_up(self.KEY_JUMP)

            # Aim
            self._win_move_mouse(aim_screen_x, aim_screen_y)

            # Fire
            if inp.fire:
                self._win_mouse_click("left")

            # Hook
            if inp.hook:
                self._win_mouse_click("right")

            # Weapon switch
            if inp.wanted_weapon in self.WEAPON_KEYS:
                vk = self.WEAPON_KEYS[inp.wanted_weapon]
                self._win_key_down(vk)
                time.sleep(0.05)
                self._win_key_up(vk)

        else:
            # Linux fallback using xdotool
            linux_map = {self.KEY_LEFT: "a", self.KEY_RIGHT: "d", self.KEY_JUMP: "space"}
            if inp.direction < 0:
                self._linux_key_down("a"); self._linux_key_up("d")
            elif inp.direction > 0:
                self._linux_key_down("d"); self._linux_key_up("a")
            else:
                self._linux_key_up("a"); self._linux_key_up("d")

            if inp.jump:
                self._linux_key_down("space")
            else:
                self._linux_key_up("space")

            self._linux_move_mouse(aim_screen_x, aim_screen_y)

    def release_all(self):
        """Release all held keys (call on bot exit)."""
        for vk in list(self._held_keys):
            self._win_key_up(vk)


# ─────────────────────────────────────────────────────────────────────────────
# BOT BASE CLASS
# Subclass this and override on_tick() to implement your bot logic.
# ─────────────────────────────────────────────────────────────────────────────

class TeeworldsBot:
    """
    Base bot class. Subclass and implement on_tick().

    Usage:
        class MyBot(TeeworldsBot):
            def on_tick(self, gs: GameState) -> PlayerInput:
                inp = PlayerInput()
                local = gs.local_player
                if local and gs.enemies:
                    enemy = gs.enemies[0]
                    # Aim at enemy
                    dx = enemy.pos.x - local.pos.x
                    dy = enemy.pos.y - local.pos.y
                    inp.target_x = int(dx)
                    inp.target_y = int(dy)
                    inp.fire = 1
                    # Walk toward enemy
                    inp.direction = 1 if dx > 0 else -1
                return inp

        bot = MyBot(use_memory=False)
        bot.run()
    """

    TICK_RATE = 50          # Teeworlds runs at 50 ticks/second
    TICK_MS   = 1000 / TICK_RATE

    def __init__(self, use_memory: bool = False, process_name: str = "teeworlds.exe"):
        self.use_memory = use_memory
        self.process_name = process_name
        self.game_state = GameState()
        self.input_sender = InputSender()
        self.reader: Optional[MemoryReader] = None
        self._running = False

        if use_memory:
            self.reader = MemoryReader(process_name)

    # Override this in your subclass
    def on_tick(self, gs: GameState) -> PlayerInput:
        """Called every game tick. Return the input to send."""
        return PlayerInput()

    # Optional override: called once after connection/attach
    def on_start(self, gs: GameState):
        pass

    # Optional override: called on each tick for debug printing
    def on_debug(self, gs: GameState, inp: PlayerInput):
        local = gs.local_player
        if local:
            print(
                f"[Tick {gs.game_tick:6d}] "
                f"pos={local.pos} hp={local.health} armor={local.armor} "
                f"weapon={local.active_weapon} | "
                f"inp=dir{inp.direction} jump={inp.jump} fire={inp.fire}"
            )

    def _world_to_screen(self, wx: float, wy: float,
                          cam_x: float, cam_y: float,
                          screen_w: int = 1920, screen_h: int = 1080,
                          zoom: float = 1.0) -> tuple:
        """Convert world coordinates to screen pixel coordinates."""
        scale = min(screen_w, screen_h) / (800.0 * zoom)
        sx = int((wx - cam_x) * scale + screen_w / 2)
        sy = int((wy - cam_y) * scale + screen_h / 2)
        return sx, sy

    def run(self, snapshot_base_addr: int = 0, char_base_addr: int = 0,
            screen_w: int = 1920, screen_h: int = 1080):
        """
        Main bot loop.

        Args:
            snapshot_base_addr: Base address of game snapshot in memory
                                 (only needed if use_memory=True).
            char_base_addr:     Base address of character array in memory
                                 (only needed if use_memory=True).
            screen_w/screen_h:  Your screen resolution for mouse aim.
        """
        if self.use_memory:
            if not self.reader.connect():
                print("[Bot] Could not attach to process. Exiting.")
                return

        self._running = True
        self.on_start(self.game_state)

        print("[Bot] Running. Press Ctrl+C to stop.")
        try:
            while self._running:
                t0 = time.perf_counter()

                # ── Read game state ──────────────────────────────────────────
                if self.use_memory and self.reader:
                    self.game_state = self.reader.read_game_state(
                        snapshot_base_addr, char_base_addr
                    )
                # (For SnapshotClient, state is updated by a background thread)

                # ── Compute bot decision ─────────────────────────────────────
                inp = self.on_tick(self.game_state)

                # ── Send input ───────────────────────────────────────────────
                local = self.game_state.local_player
                if local and inp:
                    # Convert aim target to screen coords
                    cam_x = local.pos.x
                    cam_y = local.pos.y
                    aim_x = cam_x + inp.target_x
                    aim_y = cam_y + inp.target_y
                    sx, sy = self._world_to_screen(
                        aim_x, aim_y, cam_x, cam_y, screen_w, screen_h
                    )
                    self.input_sender.apply_input(inp, sx, sy)

                # ── Debug output ─────────────────────────────────────────────
                self.on_debug(self.game_state, inp)

                # ── Tick rate throttle ───────────────────────────────────────
                elapsed_ms = (time.perf_counter() - t0) * 1000
                sleep_ms = max(0.0, self.TICK_MS - elapsed_ms)
                time.sleep(sleep_ms / 1000.0)

        except KeyboardInterrupt:
            print("\n[Bot] Stopped by user.")
        finally:
            self.input_sender.release_all()


# ─────────────────────────────────────────────────────────────────────────────
# EXAMPLE BOT IMPLEMENTATIONS
# ─────────────────────────────────────────────────────────────────────────────

class AimBot(TeeworldsBot):
    """
    Simple aimbot: aims at the nearest enemy and fires.
    Falls back to walking right if no enemies visible.
    """

    def on_tick(self, gs: GameState) -> PlayerInput:
        inp = PlayerInput()
        local = gs.local_player
        if not local or not local.alive:
            return inp

        enemies = gs.enemies
        if not enemies:
            # Wander right when no enemies
            inp.direction = 1
            inp.jump = 1
            return inp

        # Find nearest alive enemy
        def dist(p):
            dx = p.pos.x - local.pos.x
            dy = p.pos.y - local.pos.y
            return (dx*dx + dy*dy) ** 0.5

        target = min(enemies, key=dist)
        dx = target.pos.x - local.pos.x
        dy = target.pos.y - local.pos.y
        d = (dx*dx + dy*dy) ** 0.5

        # Aim
        inp.target_x = int(dx)
        inp.target_y = int(dy)

        # Move toward target
        inp.direction = 1 if dx > 0 else -1

        # Choose weapon based on distance
        if d < 100:
            inp.wanted_weapon = 0    # Hammer – close range
        elif d < 400:
            inp.wanted_weapon = 1    # Gun
        elif d < 700:
            inp.wanted_weapon = 2    # Shotgun
        else:
            inp.wanted_weapon = 4    # Laser – long range

        # Fire continuously
        inp.fire = 1

        # Jump over obstacles (simple: jump periodically)
        if gs.game_tick % 60 == 0:
            inp.jump = 1

        # Use health pickup bias – move up when low health
        if local.health < 3:
            inp.target_y = int(dy - 200)  # aim slightly upward (retreat arc)
            inp.jump = 1

        return inp


class HookBot(TeeworldsBot):
    """
    Uses hook to pull enemies then hammers them.
    """

    HOOK_RANGE = 380.0    # Teeworlds hook range in world units

    def on_tick(self, gs: GameState) -> PlayerInput:
        inp = PlayerInput()
        local = gs.local_player
        if not local or not local.alive:
            return inp

        enemies = gs.enemies
        if not enemies:
            inp.direction = 1
            return inp

        def dist(p):
            dx = p.pos.x - local.pos.x
            dy = p.pos.y - local.pos.y
            return (dx*dx + dy*dy) ** 0.5

        target = min(enemies, key=dist)
        dx = target.pos.x - local.pos.x
        dy = target.pos.y - local.pos.y
        d = dist(target)

        inp.target_x = int(dx)
        inp.target_y = int(dy)
        inp.direction = 1 if dx > 0 else -1

        if d <= self.HOOK_RANGE:
            # Hook + hammer combo
            inp.hook = 1
            inp.wanted_weapon = 0   # Hammer
            inp.fire = 1 if d < 80 else 0
        else:
            # Close distance with gun
            inp.wanted_weapon = 1
            inp.fire = 1

        return inp


# ─────────────────────────────────────────────────────────────────────────────
# MAIN – run the demo
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("Teeworlds Bot Framework")
    print("=" * 50)
    print("Available bots:")
    print("  AimBot    – aims at nearest enemy and shoots")
    print("  HookBot   – hooks enemies then hammers them")
    print()
    print("To use:")
    print("  1. Launch Teeworlds and join a server.")
    print("  2. Find the snapshot base address and character array")
    print("     address using Cheat Engine (scan for your HP value).")
    print("  3. Edit OFFSETS in MemoryReader for your build.")
    print("  4. Run:")
    print()
    print("     bot = AimBot(use_memory=True, process_name='teeworlds.exe')")
    print("     bot.run(snapshot_base_addr=0xDEADBEEF,")
    print("             char_base_addr=0xCAFEBABE,")
    print("             screen_w=1920, screen_h=1080)")
    print()
    print("Alternatively, integrate SnapshotParser with a network client")
    print("to receive live snapshots from the server without memory reading.")

    # ── Quick demo: simulate a fake game state and run AimBot for 3 ticks ──
    print("\n[Demo] Simulating 3 bot ticks with fake game state...\n")

    fake_state = GameState(game_tick=100, local_client_id=0)
    fake_state.players[0] = PlayerState(
        client_id=0, team=0,
        pos=Vec2(500, 300), vel=Vec2(0, 0),
        health=10, armor=5, alive=True,
        active_weapon=1
    )
    fake_state.players[1] = PlayerState(
        client_id=1, team=1,
        pos=Vec2(800, 290), vel=Vec2(-2, 0),
        health=7, armor=0, alive=True,
        active_weapon=2
    )

    class DemoBot(AimBot):
        """Demo version that doesn't actually send OS input."""
        tick_count = 0

        def on_tick(self, gs: GameState) -> PlayerInput:
            inp = super().on_tick(gs)
            gs.game_tick += 1
            # Inject fresh fake state each tick
            return inp

        def on_debug(self, gs: GameState, inp: PlayerInput):
            super().on_debug(gs, inp)
            self.tick_count += 1
            if self.tick_count >= 3:
                self._running = False

    demo = DemoBot(use_memory=False)
    demo.game_state = fake_state

    # Monkey-patch input sender to be a no-op for demo
    demo.input_sender.apply_input = lambda inp, sx, sy: None

    demo.run()


: 